# Neural Style Transfer on AWS SageMaker

Apply the artistic style of a reference image to a content image using a pre-trained VGG19 network, running the optimization on an AWS SageMaker training job.

**References:**
- https://github.com/udacity/deep-learning-v2-pytorch/tree/master/style-transfer
- https://elix-tech.github.io/ja/2016/08/22/art.html

In [ ]:
# import resources
%matplotlib inline

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import tarfile
import boto3
import torch
from torchvision import transforms, models

In [ ]:
# The folder we will use for storing data
data_dir = '../data/'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Use GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def load_image(img_path, max_size=400, shape=None):
    image = Image.open(img_path).convert('RGB')

    if max(image.size) > max_size:
        size = max_size
    else:
        size = max(image.size)

    if shape is not None:
        size = shape

    in_transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406),
                             (0.229, 0.224, 0.225))])
    image = in_transform(image)[:3, :, :].unsqueeze(0)
    return image

In [ ]:
# Names of the images placed in the data directory (uploaded to S3)
input_image_name = 'mychild.jpg'
reference_image_name = 'illustration.jpg'

# These validation steps run locally and ensure the image files exist before training
content_path = os.path.join(data_dir, input_image_name)
style_path = os.path.join(data_dir, reference_image_name)
print('Content image exists:', os.path.exists(content_path))
print('Style image exists:  ', os.path.exists(style_path))

In [ ]:
# Load in content and style image
content = load_image(content_path).to(device)
# Resize style to match content, makes code easier
style = load_image(style_path, shape=content.shape[-2:]).to(device)

In [ ]:
def im_convert(tensor):
    image = tensor.to("cpu").clone().detach().numpy().squeeze()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    image = image.clip(0, 1)
    return image

In [ ]:
# Display the content and style images side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(im_convert(content))
ax1.set_title('Content Image')
ax2.imshow(im_convert(style))
ax2.set_title('Style Image')
plt.show()

## 1. SageMaker session setup

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.inputs import TrainingInput

sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
prefix = 'sagemaker/style_transfer'

# Retrieve the current execution role from the notebook instance
role = sagemaker.get_execution_role()

In [ ]:
# Upload the local data directory (content + style images) to S3
input_data = sagemaker_session.upload_data(path=data_dir, bucket=bucket, key_prefix=prefix)
print('Uploaded data to:', input_data)

## 2. Create the PyTorch estimator

The training script (`scripts/train.py`) runs on a SageMaker managed training instance. We use a GPU instance (`ml.p2.xlarge`) to speed up the iterative style transfer optimization.

In [ ]:
estimator = PyTorch(entry_point="train.py",
                    source_dir="scripts",
                    role=role,
                    framework_version="2.0.1",
                    py_version="py310",
                    instance_count=1,
                    instance_type="ml.p2.xlarge",  # GPU instance
                    hyperparameters={
                        "epochs": 500,
                        "input_image_name": input_image_name,
                        "reference_image_name": reference_image_name
                    })
print(estimator)

In [ ]:
# Start the SageMaker training job
estimator.fit({"training": TrainingInput(input_data)})

## 3. Download and display the result

The training script saves `result.jpg` to the SageMaker model directory, which is packaged as `model.tar.gz` and uploaded to S3. Download it and extract the stylized image.

In [ ]:
# Download the model artifact (model.tar.gz) from S3
model_uri = estimator.model_data
print('Model artifact at:', model_uri)

bucket_name, key = model_uri.replace('s3://', '').split('/', 1)
local_tar = os.path.join(data_dir, 'model.tar.gz')

s3 = boto3.client('s3')
s3.download_file(bucket_name, key, local_tar)
print('Downloaded to:', local_tar)

In [ ]:
# Extract the result image from the downloaded archive
model_local_dir = os.path.join(data_dir, 'model')
os.makedirs(model_local_dir, exist_ok=True)
with tarfile.open(local_tar) as tar:
    tar.extractall(path=model_local_dir)

result_path = os.path.join(model_local_dir, 'result.jpg')
print('Result image exists:', os.path.exists(result_path))

In [ ]:
if os.path.exists(result_path):
    result_image = Image.open(result_path).convert('RGB')

    # Display content and final, target image
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    ax1.imshow(im_convert(content))
    ax1.set_title('Content Image')
    ax2.imshow(result_image)
    ax2.set_title('Stylized Result')
    plt.show()
else:
    print('Result image not found. Check the training job output.')